## Export clipped data to excel
Author: Lara Spearman <br>
Date: 01/09/2025 <br>
Purpose: Takes output csv from data prep.ipynb and exports it as excel file

In [85]:
from jinja2 import Environment, FileSystemLoader
import pandas as pd
import geopandas as gpd
pd.options.display.float_format = '{:,.1f}'.format
from pathlib import Path
import os

In [86]:
pc = "Lydiard Millicent CP"

In [87]:
# Get current working directory
cwd = Path.cwd()
location_data = "C:/Users/LaraS/Wiltshire Wildlife Trust/WSBRC Data - Documents/Services/Neighbourhood_Planning/_NeigbourhoodPlansPython/NeigbourhoodPlansPython.gdb"
output_folder = f"{cwd}/../Jobs/"

In [88]:
gdf = gpd.read_file(location_data, layer= "OS_BL_Parish_Wilts")

In [89]:
# Create rank for PC area
gdf['AreaPC'] = gdf['Hectares']
df_pc_sorted = gdf.sort_values(by = "AreaPC", ascending=False)
df_pc_sorted['rankArea'] = list(range(1, df_pc_sorted.shape[0]+1))

In [90]:
dfpc = df_pc_sorted[df_pc_sorted.Name == pc]
pcAreaRank =dfpc['rankArea'].item()

In [91]:
df = pd.read_csv("data/outputClippedData.csv")


df_pc = df[df.Name == pc]

In [92]:
df_top_ten =  df[df['rank']<=10]
df_top_ten.to_csv("data/topTen.csv", index=False)

In [93]:
df_pc_renamed = df_pc.rename(columns={"groupColumnValue":"Category",
                      #"value":f"{df_pc.unitName.iloc[0]} ({df_pc.unit.iloc[0]})",
                      "valuePercent": "Percentage of PC",
                       "rank": "PC area rank"})

In [ ]:
df_pc_renamed = df_pc_renamed.drop(columns = ["mapName", "AreaPC","groupColumnName",  "rankPercent"])# 
df_pc_renamed = df_pc_renamed[['Name', 'datasetName', "Category",'value','unitName', 'unit','Percentage of PC',"PC area rank",'source','mapGroup']]
df_pc_renamed = df_pc_renamed.sort_values("mapGroup")
df_pc_renamed['Percentage of PC'] = df_pc_renamed['Percentage of PC'].round(1)

In [106]:
#Create folder if not already there
if not os.path.exists(f"{output_folder}{pc}"):
    os.makedirs(f"{output_folder}{pc}")

# Write out to excel, with each grouping on a new sheet
with pd.ExcelWriter(f'{output_folder}{pc}/{pc}_outputReport.xlsx') as writer: 
    for group in df_pc_renamed.mapGroup.unique():
        mapGroup_df = df_pc_renamed[df_pc_renamed.mapGroup == group]
        
        mapGroup_df.to_excel(writer, sheet_name=group, index = False)
       